# Data Science

### Sales Forecasting (DemandSense AI)
    
    Pada notebook ini dilakukan penggabungan kekuatan LSTM/GRU dan XGBoost. Project data science ini berfokus pada efisiensi inventaris melalui prediksi 30 hari ke depan menggunakan metode Recursive Multi-step Forecasting. Dengan fitur automated preprocessing dan hybrid modeling, hasil prediksi divisualisasikan secara interaktif untuk mendukung pengambilan keputusan bisnis yang cepat dan akurat. Sehingga dapat mengoptimalkan manajemen stok dan rantai pasok, kami mengembangkan sistem prediksi penjualan cerdas berbasis data historis. Melalui integrasi teknologi Deep Learning dan Ensemble Learning, sistem ini tidak hanya memprediksi angka penjualan, tetapi juga memberikan rekomendasi jumlah stok yang akurat untuk setiap kategori produk, sehingga meminimalisir risiko stok habis (out-of-stock) maupun penumpukan barang berlebih

Author by: Risyadhana Syaifuddin & Deni Bachtiar

## 1. Import Library

In [1]:
import numpy as np
import pandas as pd
import glob
import os
import tensorflow as tf
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout, Input, BatchNormalization, Bidirectional

## 2. Load Data

In [2]:
# Load data
files = glob.glob("forecast_*_data.csv")
kategori_mapping = {
    "forecast_bathroom_data": "Bathroom", "forecast_home_data": "Home",
    "forecast_kitchen_data": "Kitchen", "forecast_storage_data": "Storage",
    "forecast_tools_data": "Tools", "forecast_other_data": "Other"
}

## 3. Preprocessing Data

In [3]:
time_steps = 30 

In [4]:
all_dfs = []

for file in files:
    fb = os.path.splitext(os.path.basename(file))[0]
    nama_kat = kategori_mapping.get(fb, fb)
    df = pd.read_csv(file)
    df['Waktu Pesanan Dibuat'] = pd.to_datetime(df['Waktu Pesanan Dibuat'])
    df = df.sort_values('Waktu Pesanan Dibuat')
    df['Net_Sales'] = (df['Jumlah'] - df['Returned Quantity']).clip(lower=0)
    
    # Fitur Lags & Moving Average
    df['lag_1'] = np.log1p(df['Net_Sales'].shift(1).fillna(0))
    df['lag_7'] = np.log1p(df['Net_Sales'].shift(7).fillna(0))
    df['lag_28'] = np.log1p(df['Net_Sales'].shift(28).fillna(0))
    df['ma_7'] = np.log1p(df['Net_Sales'].rolling(7).mean().fillna(0))
    df['day_sin'] = np.sin(2 * np.pi * df['Waktu Pesanan Dibuat'].dt.day / 31)
    df['day_cos'] = np.cos(2 * np.pi * df['Waktu Pesanan Dibuat'].dt.day / 31)
    df['Kategori'] = nama_kat
    
    # Over-sampling kategori kecil
    if nama_kat in ['Bathroom', 'Storage', 'Other']:
        df = pd.concat([df] * 3, ignore_index=True)
    all_dfs.append(df.dropna())

## 4. Splitting Data & Transformasi Data

### 4.1 Data Aggregation 

    Pada tahap ini dilakukan menyatukan semua sumber data menjadi satu kesatuan (dataframe tunggal).

In [5]:
# inisiasi untuk gabungin semua list dataframe menjadi satu
full_df = pd.concat(all_dfs, ignore_index=True)

# Inisiasi buat nampilin hasil penggabungan
display(full_df.head())

,Waktu Pesanan Dibuat,Jumlah,Returned Quantity,Total Pembayaran,Total Diskon,Ongkos Kirim Dibayar oleh Pembeli,Jumlah Terjual Bersih,Weekend,Net_Sales,lag_1,lag_7,lag_28,ma_7,day_sin,day_cos,Kategori
0,2023-12-04,3,0,110840,0.0,16000.0,3,0.0,3,0.000000,0.0,0.0,0.0,0.724793,0.688967,Bathroom
1,2023-12-05,1,0,18680,0.0,0.0,1,0.0,1,1.386294,0.0,0.0,0.0,0.848644,0.528964,Bathroom
2,2023-12-06,0,0,0,0.0,0.0,0,0.0,0,0.693147,0.0,0.0,0.0,0.937752,0.347305,Bathroom
3,2023-12-07,0,0,0,0.0,0.0,0,0.0,0,0.000000,0.0,0.0,0.0,0.988468,0.151428,Bathroom
4,2023-12-08,0,0,0,0.0,0.0,0,0.0,0,0.000000,0.0,0.0,0.0,0.998717,-0.050649,Bathroom


### 4.2 Categorical Encoding (Transformasi Data Kategori)

    Selanjutnya pada tahap ini dilakukan pengubahan kolom teks 'Kategori' menjadi bentuk angka biner supaya bisa diproses model.

In [6]:
# Inisialisasi OneHotEncoder
encoder = OneHotEncoder(sparse_output=False)

# inisiasi kat_encoded untuk fit dan transform kolom kategori
kat_encoded = encoder.fit_transform(full_df[['Kategori']])

# inisiasi buat nama kolom baru untuk hasil encoding (misal: Cat_Elektronik, Cat_Fashion)
kat_cols = [f"Cat_{c}" for c in encoder.categories_[0]]

# inisiasi full_df untuk ngegabungin lagi hasil encoding ke dataframe utama
full_df = pd.concat([full_df.reset_index(drop=True), 
                     pd.DataFrame(kat_encoded, columns=kat_cols)], axis=1)

print("Data Setelah One-Hot Encoding")
display(full_df[kat_cols].head())

Data Setelah One-Hot Encoding


,Cat_Bathroom,Cat_Home,Cat_Kitchen,Cat_Other,Cat_Storage,Cat_Tools
0,1.0,0.0,0.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0,0.0,0.0
2,1.0,0.0,0.0,0.0,0.0,0.0
3,1.0,0.0,0.0,0.0,0.0,0.0
4,1.0,0.0,0.0,0.0,0.0,0.0


### 4.3 Feature Selection & Target Transformation (Rekayasa Fitur & Target)

    Selanjutnya pada proses ini, dilakukan penentuan fitur apa yang akan dipakai dan menormalisasi target supaya distribusinya lebih bagus (mengurangi skewness).

In [7]:
# inisiasi untuk mendefinisikan list fitur numerik dan kategori yang akan digunakan model
num_features = ['day_sin', 'day_cos', 'lag_1', 'lag_7', 'lag_28', 'ma_7']
features = num_features + kat_cols

# insiasi untuk transformasi Log pada target (Net_Sales) sehingga dapat menstabilkan varians
# Rumus: log(1 + x) supaya aman dari nilai nol
full_df['Target_Log'] = np.log1p(full_df['Net_Sales'])

print("\n List Fitur yang Digunakan")
print(features)
print("\n Data Final (Beberapa Kolom)")
display(full_df[features + ['Target_Log']].head())


 List Fitur yang Digunakan
['day_sin', 'day_cos', 'lag_1', 'lag_7', 'lag_28', 'ma_7', 'Cat_Bathroom', 'Cat_Home', 'Cat_Kitchen', 'Cat_Other', 'Cat_Storage', 'Cat_Tools']

 Data Final (Beberapa Kolom)


,day_sin,day_cos,lag_1,lag_7,lag_28,ma_7,Cat_Bathroom,Cat_Home,Cat_Kitchen,Cat_Other,Cat_Storage,Cat_Tools,Target_Log
0,0.724793,0.688967,0.000000,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.386294
1,0.848644,0.528964,1.386294,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.693147
2,0.937752,0.347305,0.693147,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.000000
3,0.988468,0.151428,0.000000,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.000000
4,0.998717,-0.050649,0.000000,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.000000


### 4.4 Feature Scaling (Standardization)

    Pada tahap ini dilakukan scaling dengan menggunakan StandardScaler supaya fitur-fitur numerik memiliki rata-rata dan standar deviasi

In [8]:
scaler = StandardScaler()
# insiasi untuk engubah angka asli (lag, ma, sin, cos) ke skala standar
full_df[num_features] = scaler.fit_transform(full_df[num_features])

print("Data Setelah Scaling")
display(full_df[num_features].head())

Data Setelah Scaling


,day_sin,day_cos,lag_1,lag_7,lag_28,ma_7
0,1.006286,1.017671,-0.926803,-0.919944,-0.898069,-1.239979
1,1.179863,0.789161,0.303679,-0.919944,-0.898069,-1.239979
2,1.304747,0.529722,-0.311562,-0.919944,-0.898069,-1.239979
3,1.375825,0.249977,-0.926803,-0.919944,-0.898069,-1.239979
4,1.390188,-0.038622,-0.926803,-0.919944,-0.898069,-1.239979


### 4.5 Time-Series Sliding Window & Weighting

    Pada tahap ini dilakukan pengubahan data yang tadinya baris-per-baris menjadi potongan-potongan waktu (sequences). Sehingga bisa dibilang dilakukan penambahan bobot (weight) pada kategori tertentu yang dianggap lebih penting

In [9]:
X_list, y_list, weights_list = [], [], []
test_meta = {}

for kat in full_df['Kategori'].unique():
    temp_df = full_df[full_df['Kategori'] == kat].copy()
    if len(temp_df) <= time_steps: continue
    
    f_vals = temp_df[features].values
    t_vals = temp_df['Target_Log'].values
    X_k, y_k = [], []
    
    # looping untuk membuat Sliding Window (ibaratnya pake data 7 hari sebelumnya untuk prediksi hari ini)
    for i in range(time_steps, len(temp_df)):
        X_k.append(f_vals[i-time_steps:i, :])
        y_k.append(t_vals[i])
    
    # insiasi untuk Train-Test Split (80% Train, 20% Test) per Kategori
    split = int(len(X_k) * 0.8)
    X_list.extend(X_k[:split])
    y_list.extend(y_k[:split])
    
    # inisiasi untuk Sample Weighting dengan nambahin bobot 5x lebih besar untuk kategori Storage & Bathroom
    w = 5.0 if kat in ['Storage', 'Bathroom'] else 1.0
    weights_list.extend([w] * split)
    
    # inisiasi untuk simpan data testing beserta nilai aslinya (setelah dibalikin dari Log)
    test_meta[kat] = {
        'X_test': np.array(X_k[split:]), 
        'y_act': np.expm1(t_vals[time_steps:][split:])
    }

### 4.6 Final Array Conversion

    Pada tahap ini dilakukan pengubahan list Python menjadi format NumPy Array supaya bisa langsung diterima oleh TensorFlow/Keras. 

In [10]:
# konversi ke format array 3D (samples, time_steps, features)
X_train = np.array(X_list)
y_train = np.array(y_list)
weights_train = np.array(weights_list)

print(f"Shape X_train: {X_train.shape}") # Outputnya berupa Jumlah Data, Time Steps, dan Jumlah Fitur
print(f"Shape y_train: {y_train.shape}")

Shape X_train: (6797, 30, 12)
Shape y_train: (6797,)


## 5. Model Architecture & Training (Modelling)

    Pada tahapan ini dilakukan hybrid feature extraction & stacking ensamble

### Initialization & Reproducibility Setting

In [ ]:
tf.keras.backend.clear_session()
np.random.seed(42)
tf.random.set_seed(42)

### 5.1 Arsitektur Hybrid RNN (Feature Extractor)

    Setelah itu dilakukan Feature Extractor (Neural Representation), bisa dibilang tidak dilakukan pengambilan output prediksi (angka 1), tapi ngambil isi "otak" atau main model di layer 'feature_layer'. Dengan tujuannya untuk mengubah urutan waktu yang rumit menjadi 32 angka representasi (vektor)



    Why?? Karena bisa dibilang nantinya XGBoost biasanya kesulitan baca data 3D mentah, tapi sangat jago baca data vektor hasil ekstraksi neural network.

    Sehingga pada tahap ini terdapat Hybrid RNN (LSTM + GRU) yang memiliki beberapa fungsi, diantaranya:

1. Bidirectional LSTM: untuk membaca data dari depan-ke-belakang dan belakang-ke-depan supaya tidak ada pola yang terlewat

2. BatchNormalization: untuk menstabilkan training agar tidak meledak (karena kamu pakai aktivasi Swish dan ReLU di tempat lain)

3. Swish Activation: dilakukan aktivasi modern (bisa dibilang yang sering dipakai di EfficientNet) dan biasanya lebih bagus daripada ReLU untuk model yang dalam (tricky)

4. Huber Loss: dilakukan Huber Loss lebih tahan terhadap outliers dibandingkan MSE

In [11]:
def build_finisher_model(input_shape):
    inputs = Input(shape=input_shape)
    
    # Bidirectional LSTM
    x = Bidirectional(LSTM(64, return_sequences=True))(inputs)
    
    # BatchNormalization
    x = BatchNormalization()(x)
    
    # GRU: Layer recurrent yang lebih ringan untuk menangkap dependensi jangka panjang
    x = GRU(64)(x)
    
    # Dropout untuk mencegah overfitting dengan mematikan 30% neuron secara acak saat training
    x = Dropout(0.3)(x)
    
    # Feature Layer: Inilah "inti" saripati data yang akan diambil nanti (32 dimensi)
    # dengan menggunakan 'swish' (aktivasi modern yang lebih smooth dibanding ReLU)
    x = Dense(32, activation='swish', name='feature_layer')(x)
    
    # Output Layer: Output tunggal untuk nilai regresi
    outputs = Dense(1)(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    
    # Optimizer & Loss: Huber Loss digunakan karena lebih kebal terhadap data pencilan (outliers)
    model.compile(optimizer=tf.keras.optimizers.Adam(0.0004), loss=tf.keras.losses.Huber())
    return model

# Inisialisasi model dengan dimensi (7 hari, jumlah_fitur)
model_rnn = build_finisher_model((time_steps, len(features)))

### 5.2 Training Fase 1 (Deep Learning)

    Pada tahap ini dilakukan train RNN supaya bisa mengenali karakteristik penjualan tiap kategori.

In [12]:
# inisiasi model_rnn
model_rnn.fit(X_train, y_train, epochs=50, batch_size=64, verbose=1)

Epoch 1/50
107/107 ━━━━━━━━━━━━━━━━━━━━ 17s 64ms/step - loss: 0.3275
Epoch 2/50
107/107 ━━━━━━━━━━━━━━━━━━━━ 6s 56ms/step - loss: 0.2695
Epoch 3/50
107/107 ━━━━━━━━━━━━━━━━━━━━ 6s 55ms/step - loss: 0.2605
Epoch 4/50
107/107 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step - loss: 0.2516
Epoch 5/50
107/107 ━━━━━━━━━━━━━━━━━━━━ 9s 88ms/step - loss: 0.2465
Epoch 6/50
107/107 ━━━━━━━━━━━━━━━━━━━━ 8s 78ms/step - loss: 0.2422
Epoch 7/50
107/107 ━━━━━━━━━━━━━━━━━━━━ 7s 67ms/step - loss: 0.2443
Epoch 8/50
107/107 ━━━━━━━━━━━━━━━━━━━━ 8s 76ms/step - loss: 0.2419
Epoch 9/50
107/107 ━━━━━━━━━━━━━━━━━━━━ 8s 70ms/step - loss: 0.2375
Epoch 10/50
107/107 ━━━━━━━━━━━━━━━━━━━━ 7s 61ms/step - loss: 0.2379
Epoch 11/50
107/107 ━━━━━━━━━━━━━━━━━━━━ 7s 68ms/step - loss: 0.2340
Epoch 12/50
107/107 ━━━━━━━━━━━━━━━━━━━━ 7s 62ms/step - loss: 0.2332
Epoch 13/50
107/107 ━━━━━━━━━━━━━━━━━━━━ 7s 61ms/step - loss: 0.2310
Epoch 14/50
107/107 ━━━━━━━━━━━━━━━━━━━━ 7s 66ms/step - loss: 0.2291
Epoch 15/50
107/107 ━━━━━━━━━━━━━━━━━━━━ 6

### 5.3 Neural Feature Extraction

    Pada tahap ini, dilakukan cutting model di tengah untuk mengambil fiturnya saja, sehingga tidak memakai hasil prediksi RNN. Dikarenakan XGBoost tidak bisa membaca data 3D (Sequence), bisa dibilang data sequence sudah "diringkas" menjadi 32 angka sakti oleh RNN supaya bisa dibaca XGBoost

In [13]:
# inisiasi untuk membuat model baru yang berhenti di 'feature_layer'
feature_extractor = Model(inputs=model_rnn.input, outputs=model_rnn.get_layer('feature_layer').output)

# inisiasi untuk mengubah data X_train (3D) menjadi vektor fitur (2D) berkekuatan 32 dimensi
X_train_feats = feature_extractor.predict(X_train, verbose=1)

213/213 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step


### 5.4 Training Fase 2 (Dual-Tweedie Ensemble)

    Pada tahap ini dilakukan penggabungan dua model XGBoost dengan parameter distribusi Tweedie yang berbeda

In [14]:
# inisiasi model 1 untuk fokus pada volume (Variance Power rendah mendekati distribusi Poisson)
xgb_vol = xgb.XGBRegressor(objective='reg:tweedie', tweedie_variance_power=1.2, n_estimators=500)

# inisiasi model 2 untuk fokus pada variansi tinggi/nilai besar (Variance Power tinggi mendekati Gamma)
xgb_mape = xgb.XGBRegressor(objective='reg:tweedie', tweedie_variance_power=1.8, n_estimators=500)

# train kedua model dengan fitur hasil ekstrak RNN dan bobot kategori (weights_train)
xgb_vol.fit(X_train_feats, y_train, sample_weight=weights_train)
xgb_mape.fit(X_train_feats, y_train, sample_weight=weights_train)

,objective,'reg:tweedie'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


## 6. Model Evaluation

    Pada tahap ini dilakukan pengujian seberapa akurat hybrid RNN-XGBoost pada data yang belum pernah dilihat model selama training

### 6.1 Inference & Evaluation (Backtesting)

In [30]:
final_report_data = [] # Inisialisasi awal
last_date = full_df['Waktu Pesanan Dibuat'].max()
eval_results = {} 

for kat, data in test_meta.items():
    # Prediksi Dasar
    t_feats = feature_extractor.predict(data['X_test'], verbose=0)
    p_v = np.expm1(xgb_vol.predict(t_feats))
    p_m = np.expm1(xgb_mape.predict(t_feats))
    
    # Ensemble Logic
    if kat in ['Kitchen', 'Home']:
        preds = np.ceil((0.8 * p_v) + (0.2 * p_m))
    else:
        preds = (0.3 * p_v) + (0.7 * p_m)
        preds = np.where(preds < 0.25, 0, np.ceil(preds))
        
    act = data['y_act']
    mae = np.mean(np.abs(act - preds))
    mape = np.mean(np.abs(act - preds) / np.maximum(act, 1)) * 100
    vol_acc = (1 - (np.abs(np.sum(act) - np.sum(preds)) / (np.sum(act) + 1e-7))) * 100
    
    # Simpan data evaluasi sementara
    eval_results[kat] = {
        'mae': mae, 'mape': mape, 'vol_acc': vol_acc, 
        'act_total': np.sum(act), 'pred_total': np.sum(preds)
    }

### 6.2 Recursive 30-Day Forecasting

In [31]:
forecast_results = {}

for kat in test_meta.keys():
    curr_df = full_df[full_df['Kategori'] == kat].sort_values('Waktu Pesanan Dibuat').tail(time_steps)
    curr_win = curr_df[features].values.tolist()
    total_forecast_30d = 0
    
    for i in range(1, 31):
        X_in = np.array([curr_win[-time_steps:]])
        lat = feature_extractor.predict(X_in, verbose=0)
        pv_f = np.expm1(xgb_vol.predict(lat))[0]
        pm_f = np.expm1(xgb_mape.predict(lat))[0]
        
        if kat in ['Kitchen', 'Home']:
            d_p = np.ceil((0.8 * pv_f) + (0.2 * pm_f))
        else:
            d_p = (0.3 * pv_f) + (0.7 * pm_f)
            d_p = 0 if d_p < 0.25 else np.ceil(d_p)
        
        total_forecast_30d += d_p
        
        # Update Window
        nxt_d = last_date + pd.Timedelta(days=i)
        new_row = [
            np.sin(2*np.pi*nxt_d.day/31), np.cos(2*np.pi*nxt_d.day/31),
            np.log1p(d_p), 
            curr_win[-7][2] if len(curr_win)>=7 else 0,
            curr_win[-28][2] if len(curr_win)>=28 else 0,
            np.mean([r[2] for r in curr_win[-7:]])
        ]
        new_row.extend(curr_df[kat_cols].iloc[0].tolist())
        curr_win.append(new_row)
    
    forecast_results[kat] = total_forecast_30d

### 6.3 Data Aggregation & Final Reporting

     Pada tahap ini dilakukan rangkuman semua hasil perhitungan ke dalam format yang mudah dibaca (List of Dictionaries)

In [32]:
final_report_data = []

for kat in eval_results.keys():
    res = eval_results[kat]
    final_report_data.append({
        "Kategori": kat,
        "MAE (Unit)": round(res['mae'], 2),
        "MAPE (%)": round(res['mape'], 2),
        "Vol Acc (%)": round(max(0, res['vol_acc']), 2),
        "Asli (Total)": int(res['act_total']),
        "Pred (Total)": int(res['pred_total']),
    })

report_df = pd.DataFrame(final_report_data)
print("\n" + "="*90)
print(report_df.to_string(index=False))
print("="*90)


Kategori  MAE (Unit)  MAPE (%)  Vol Acc (%)  Asli (Total)  Pred (Total)
Bathroom        0.06      3.81        91.35           312           339
    Home        5.54     73.48        86.70          1278          1108
 Kitchen       17.27     93.41        96.64          4289          4145
   Other        0.08      3.16        97.89           568           580
 Storage        0.30      8.74        94.68          1823          1920
   Tools        8.07     69.12        85.99          1599          1375


## 7. Recursive Multi-Step Forecasting (30 Days)

### 7.1 30-Day Recursive Forecast (Hanya Prediksi)

In [33]:
all_forecast_series = {}


for kat in test_meta.keys():
    # Ambil window terakhir
    curr_df = full_df[full_df['Kategori'] == kat].sort_values('Waktu Pesanan Dibuat').tail(time_steps)
    curr_win = curr_df[features].values.tolist()
    
    daily_forecasts = [] # Untuk grafik (isi 30 angka)
    
    for i in range(1, 31):
        X_in = np.array([curr_win[-time_steps:]])
        lat = feature_extractor.predict(X_in, verbose=0)
        
        pv_f = np.expm1(xgb_vol.predict(lat))[0]
        pm_f = np.expm1(xgb_mape.predict(lat))[0]
        
        # Ensemble Logic
        if kat in ['Kitchen', 'Home']:
            d_p = np.ceil((0.8 * pv_f) + (0.2 * pm_f))
        else:
            d_p = (0.3 * pv_f) + (0.7 * pm_f)
            d_p = 0 if d_p < 0.25 else np.ceil(d_p)
        
        daily_forecasts.append(d_p)
        
        # Update Feature Window
        nxt_d = last_date + pd.Timedelta(days=i)
        new_row = [
            np.sin(2*np.pi*nxt_d.day/31), np.cos(2*np.pi*nxt_d.day/31),
            np.log1p(d_p), 
            curr_win[-7][2] if len(curr_win)>=7 else 0,
            curr_win[-28][2] if len(curr_win)>=28 else 0,
            np.mean([r[2] for r in curr_win[-7:]])
        ]
        new_row.extend(curr_df[kat_cols].iloc[0].tolist())
        curr_win.append(new_row)
    
    all_forecast_series[kat] = daily_forecasts

### 7.2 Rekomendasi Stock(30 hari)

In [38]:
stock_recommendations = []

for kat, series in all_forecast_series.items():
    total_unit = int(sum(series))
    stock_recommendations.append({
        "Kategori": kat,
        "Kebutuhan Stok (30 Hari)": total_unit,
    })

# Cetak Tabel
print("\n" + "="*50)
print(f"{'REKOMENDASI PERSIAPAN STOK':^50}")
print("="*50)
stock_df = pd.DataFrame(stock_recommendations)
print(stock_df.to_string(index=False))
print("="*50)


            REKOMENDASI PERSIAPAN STOK            
Kategori  Kebutuhan Stok (30 Hari)
Bathroom                         0
    Home                       626
 Kitchen                       730
   Other                        20
 Storage                        22
   Tools                       477


### 7.3 Visualisasi (Historical + Forecast)

In [34]:
import plotly.graph_objects as go
import pandas as pd

In [35]:
def plot_forecast_plotly(kategori_name):
    # inisiasi untuk ambil data historis 30 hari terakhir
    hist_data = full_df[full_df['Kategori'] == kategori_name].sort_values('Waktu Pesanan Dibuat').tail(30)
    hist_vals = hist_data['Net_Sales'].values
    hist_dates = pd.to_datetime(hist_data['Waktu Pesanan Dibuat']).values
    
    # inisiasi persiapan data forecast 30 hari ke depan
    start_date = hist_dates[-1]
    forecast_dates = [start_date + pd.Timedelta(days=i) for i in range(1, 31)]
    forecast_vals = all_forecast_series[kategori_name]
    
    # Inisialisasi Figure Plotly
    fig = go.Figure()

    # garis historis (biru)
    fig.add_trace(go.Scatter(
        x=hist_dates, 
        y=hist_vals,
        mode='lines+markers',
        name='Data Historis (30 Hari Terakhir)',
        line=dict(color='#1f77b4', width=3),
        marker=dict(size=6),
        hovertemplate='<b>Tanggal</b>: %{x}<br><b>Sales</b>: %{y} unit<extra></extra>'
    ))

    # garis forecast (orange)
    # inisiasi untuk menambahkan titik terakhir historis supaya garis menyambung sempurna
    conn_dates = [hist_dates[-1]] + forecast_dates
    conn_vals = [hist_vals[-1]] + forecast_vals

    fig.add_trace(go.Scatter(
        x=conn_dates, 
        y=conn_vals,
        mode='lines+markers',
        name='Prediksi 30 Hari Depan',
        line=dict(color='#ff7f0e', width=3, dash='dash'), # Garis putus-putus
        marker=dict(size=6, symbol='square'),
        hovertemplate='<b>Tanggal</b>: %{x}<br><b>Prediksi</b>: %{y} unit<extra></extra>'
    ))

    # LAYOUT & STYLING
    fig.update_layout(
        title=f"Trend Penjualan & Forecast: Kategori {kategori_name}",
        xaxis_title="Tanggal",
        yaxis_title="Unit Terjual",
        hovermode="x unified", # Menampilkan info kedua garis saat hover di satu titik X
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(l=20, r=20, t=60, b=20)
    )

    fig.show()

# inisiasi menampilkan grafik untuk semua kategori
for kat in all_forecast_series.keys():
    plot_forecast_plotly(kat)

## 8. Model Saving

In [36]:
import joblib

# simpan Scaler & Encoder untuk preprocessing
joblib.dump(scaler, 'scaler.joblib')
joblib.dump(encoder, 'encoder.joblib')
joblib.dump(features, 'feature_names.joblib')
joblib.dump(kat_cols, 'kat_cols.joblib')

# simpan Model XGBoost (Dua-duanya)
joblib.dump(xgb_vol, 'xgb_vol.joblib')
joblib.dump(xgb_mape, 'xgb_mape.joblib')

# simpan Model RNN (Format .h5) 
# Simpan model utuh
model_rnn.save('model_rnn.h5')

# Simpan Feature Extractor saja (untuk efisiensi di Inference)
feature_extractor.save('feature_extractor.h5')